In [59]:
%env DAF_BUTLER_REPOSITORY_INDEX=/global/cfs/cdirs/lsst/production/gen3/shared/data-repos.yaml
import h5py
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table, vstack
import tqdm
from astropy.io import fits
import healsparse as hsp
import lsst.geom as geom
from lsst.daf.butler import Butler

env: DAF_BUTLER_REPOSITORY_INDEX=/global/cfs/cdirs/lsst/production/gen3/shared/data-repos.yaml


# Load in catalog data

### Data Provenance:
eRASS 1: https://erosita.mpe.mpg.de/dr1/AllSkySurveyData_dr1/Catalogues_dr1/

MCXC-II: https://cdsarc.cds.unistra.fr/viz-bin/cat/J/A+A/688/A187

SPT ECS: https://pole.uchicago.edu/public/data/sptsz-clusters/

SPT SZ: https://pole.uchicago.edu/public/data/sptsz-clusters/

ACT DR6: https://lambda.gsfc.nasa.gov/product/act/actadv_dr6_szcluster_catalog_get.html

Planck DR2: https://irsa.ipac.caltech.edu/data/Planck/release_2/catalogs

DES Y3 RedMaPPer: https://des.ncsa.illinois.edu/releases/y3a2/Y3key-cluster

DES Y6 WaZP: https://datasets.linea.org.br/wazp/y6a2_dnf_wazp_v5.0.12.6801_clusters.fits

HSC Camira: https://github.com/oguri/cluster_catalogs/tree/main/hsc_s23b_camira

DECADE: https://dhayaaanbajagane.github.io/data_release/decade_clusters

In [2]:
data_path = "/global/cfs/cdirs/desc-cl/DP2_known_clusters/data"

erass = Table.read(f'{data_path}/ERASS1cl_primary_v3.2.fits', format='fits')
mcxc = Table.read(f'{data_path}/MCXC-II.dat.gz.fits', format='fits')
spt_ecs = Table.read(f'{data_path}/SPTECS_sptecs_catalog_oct919.fits', format='fits')
spt_sz = Table.read(f'{data_path}/SPTSZ_2500d_cluster_sample_Bocquet19.fits', format='fits')
act = Table.read(f'{data_path}/ACTDR6_DR6_cluster-catalog_v1.0.fits', format='fits')
planck = Table.read(f'{data_path}/PLANCK_HFI_PCCS_SZ-union_R2.08.fits', format='fits')
des_y6 = Table.read(f'{data_path}/DESY6_y6a2_dnf_wazp_v5.0.12.6801_clusters.fits', format='fits')
with h5py.File(f'{data_path}/DESY3_y3_redmapper_v6.4.22+2_release.h5', 'r') as f:
    cluster_group = f["catalog/cluster"]
    table_data = {col: cluster_group[col][:] for col in cluster_group.keys()}
    des_y3 = Table(table_data)
hsc_deep = Table.read(f'{data_path}/HSCY3_camira_s23b_dud_sm.tbl', format='ascii', 
                      names=["ID", "name", "RA", "DEC", "Z", "RICHNESS", "Z_BCG"])
hsc_wide = Table.read(f'{data_path}/HSCY3_camira_s23b_wide_sm.tbl', format='ascii', 
                      names=["ID", "name", "RA", "DEC", "Z", "RICHNESS", "Z_BCG"])
with h5py.File(f'{data_path}/DECADE_RedmapperCatalog.hdf5', 'r') as f:
    cluster_group = f["DECADE/catalog/lambda5"]
    table_data = {col: cluster_group[col][:] for col in cluster_group.keys()}
    decade = Table(table_data)

# Generate temporary tables

In [64]:
erass_tab = Table({
    'RA': erass['RA'],
    'DEC': erass['DEC'],
    'redshift': erass['BEST_Z'],
    'redshift_type': erass['BEST_Z_TYPE'],
    'redshift_unc': erass['BEST_ZERR'],
    'bcg_specz': [-1] * len(erass),
    'richness': [-1] * len(erass),
    'richness_unc': [-1] * len(erass),
    'M200m': [-1] * len(erass),
    'M500c': erass['M500'] * 1e13, # Convert from 1e13Msun
    'catalog_source': ['eRASS 1'] * len(erass),
    'catalog_id': erass['NAME'].astype(str),
    })

In [65]:
spt_ecs_tab = Table({
    'RA': spt_ecs['RA'],
    'DEC': spt_ecs['DEC'],
    'redshift': spt_ecs['REDSHIFT'],
    'redshift_type': ['-1'] * len(spt_ecs),
    'redshift_unc': spt_ecs['REDSHIFT_UNC'],
    'bcg_specz': spt_ecs['SPECZ'],
    'richness': spt_ecs['LAMBDA_CHISQ'],
    'richness_unc': spt_ecs['LAMBDA_CHISQ_E'],
    'M200m': [-1] * len(spt_ecs),
    'M500c': spt_ecs['M500'] * 1.43e14, # Convert from 1e14Msun/h
    'catalog_source': ['SPT ECS'] * len(spt_ecs),
    'catalog_id': spt_ecs['SPT_ID'].astype(str),
    })

In [66]:
spt_sz_tab = Table({
    'RA': spt_sz['RA'],
    'DEC': spt_sz['DEC'],
    'redshift': spt_sz['REDSHIFT'],
    'redshift_type': ['-1'] * len(spt_sz),
    'redshift_unc': spt_sz['REDSHIFT_UNC'],
    'bcg_specz': spt_sz['SPECZ'],
    'richness': [-1] * len(spt_sz),
    'richness_unc': [-1] * len(spt_sz),
    'M200m': [-1] * len(spt_sz),
    'M500c': spt_sz['M500'] * 1.43e14, # Convert from 1e14Msun/h
    'catalog_source': ['SPT SZ_2500d'] * len(spt_sz),
    'catalog_id': spt_sz['SPT_ID'].astype(str),
    })

In [67]:
mcxc_tab = Table({
    'RA': mcxc['RAdeg'],
    'DEC': mcxc['DEdeg'],
    'redshift': mcxc['z'],
    'redshift_type': mcxc['zType'],
    'redshift_unc': [-1] * len(mcxc),
    'bcg_specz': [-1] * len(mcxc),
    'richness': [-1] * len(mcxc),
    'richness_unc': [-1] * len(mcxc),
    'M200m': [-1] * len(mcxc),
    'M500c': mcxc['M500'] * 1e14, # Convert from 1e14Msun
    'catalog_source': ['MCXC-II'] * len(mcxc),
    'catalog_id': mcxc['Index'].astype(str),
    })

In [68]:
act_tab = Table({
    'RA': act['RADeg'],
    'DEC': act['decDeg'],
    'redshift': act['redshift'],
    'redshift_type': act['redshiftType'],
    'redshift_unc': act['redshiftErr'],
    'bcg_specz': [-1] * len(act),
    'richness': [-1] * len(act),
    'richness_unc': [-1] * len(act),
    'M200m': act['M200m'] * 1e14,
    'M500c': act['M500c'] * 1e14,
    'catalog_source': ['ACT DR6'] * len(act),
    'catalog_id': act['name'].astype(str),
    })

In [69]:
planck_tab = Table({
    'RA': planck['RA'],
    'DEC': planck['DEC'],
    'redshift': planck['REDSHIFT'],
    'redshift_type': ['-1'] * len(planck),
    'redshift_unc': [-1] * len(planck),
    'bcg_specz': [-1] * len(planck),
    'richness': [-1] * len(planck),
    'richness_unc': [-1] * len(planck),
    'M200m': [-1] * len(planck),
    'M500c': planck['MSZ'] * 1e14,
    'catalog_source': ['Planck'] * len(planck),
    'catalog_id': planck['INDEX'].astype(str),
    })

In [70]:
des_y3_tab = Table({
    'RA': des_y3['ra'],
    'DEC': des_y3['dec'],
    'redshift': des_y3['z_lambda'],
    'redshift_type': ['-1'] * len(des_y3),
    'redshift_unc': des_y3['z_lambda_e'],
    'bcg_specz': des_y3['bcg_spec_z'],
    'richness': des_y3['lambda_chisq'],
    'richness_unc': des_y3['lambda_chisq_e'],
    'M200m': des_y3['m200m'],
    'M500c': [-1] * len(des_y3),
    'catalog_source': ['DES Y3 RedMaPPer'] * len(des_y3),
    'catalog_id': des_y3['mem_match_id'].astype(str),
    })

In [71]:
des_y6_zcol = np.copy(des_y6['ZPHOT'])
des_y6_zcol[~des_y6['ZSPEC'].mask] = des_y6['ZSPEC'][~des_y6['ZSPEC'].mask]
des_y6_zunc = np.copy(des_y6['ZSPEC_UNC'])
des_y6_zunc[des_y6['ZSPEC'].mask] = -1
des_y6_ztype = np.array(['phot'] * len(des_y6))
des_y6_ztype[~des_y6['ZSPEC'].mask] = 'spec'

des_y6_tab = Table({
    'RA': des_y6['RA'],
    'DEC': des_y6['DEC'],
    'redshift': des_y6_zcol,
    'redshift_type': des_y6_ztype,
    'redshift_unc': des_y6_zunc,
    'bcg_specz': [-1] * len(des_y6),
    'richness': des_y6['NGALS'],
    'richness_unc': des_y6['NGALS_UNC'],
    'M200m': [-1] * len(des_y6),
    'M500c': [-1] * len(des_y6),
    'catalog_source': ['DES Y6 WaZP'] * len(des_y6),
    'catalog_id': des_y6['ID'].astype(str),
    })

In [72]:
decade_tab = Table({
    'RA': decade['ra'],
    'DEC': decade['dec'],
    'redshift': decade['z_lambda'],
    'redshift_type': ['-1'] * len(decade),
    'redshift_unc': decade['z_lambda_e'],
    'bcg_specz': [-1] * len(decade),
    'richness': decade['lambda'],
    'richness_unc': decade['lambda_e'],
    'M200m': [-1] * len(decade),
    'M500c': [-1] * len(decade),
    'catalog_source': ['DECADE'] * len(decade),
    'catalog_id': decade['mem_match_id'].astype(str),
    })

In [73]:
hsc_deep_tab = Table({
    'RA': hsc_deep['RA'],
    'DEC': hsc_deep['DEC'],
    'redshift': hsc_deep['Z'],
    'redshift_type': ['-1'] * len(hsc_deep),
    'redshift_unc': [-1] * len(hsc_deep),
    'bcg_specz': hsc_deep['Z_BCG'],
    'richness': hsc_deep['RICHNESS'],
    'richness_unc': [-1] * len(hsc_deep),
    'M200m': [-1] * len(hsc_deep),
    'M500c': [-1] * len(hsc_deep),
    'catalog_source': ['HSC Deep'] * len(hsc_deep),
    'catalog_id': hsc_deep['ID'].astype(str),
    })

In [74]:
hsc_wide_tab = Table({
    'RA': hsc_wide['RA'],
    'DEC': hsc_wide['DEC'],
    'redshift': hsc_wide['Z'],
    'redshift_type': ['-1'] * len(hsc_wide),
    'redshift_unc': [-1] * len(hsc_wide),
    'bcg_specz': hsc_wide['Z_BCG'],
    'richness': hsc_wide['RICHNESS'],
    'richness_unc': [-1] * len(hsc_wide),
    'M200m': [-1] * len(hsc_wide),
    'M500c': [-1] * len(hsc_wide),
    'catalog_source': ['HSC Wide'] * len(hsc_wide),
    'catalog_id': hsc_wide['ID'].astype(str),
    })

# Merge tables and set units

In [75]:
full_clusters_catalog = vstack([act_tab, 
                                decade_tab, 
                                des_y3_tab, 
                                des_y6_tab,
                                erass_tab,
                                hsc_wide_tab,
                                hsc_deep_tab,
                                mcxc_tab,
                                planck_tab,
                                spt_ecs_tab,
                                spt_sz_tab,
                               ])

In [76]:
full_clusters_catalog['catalog_id'].format = None
full_clusters_catalog['catalog_id'].unit = None
full_clusters_catalog['redshift'].unit = None
full_clusters_catalog['M200m'].unit = 'Msun'
full_clusters_catalog['M500c'].unit = 'Msun'

In [89]:
del act_tab
del decade_tab
del des_y3_tab
del des_y6_tab
del erass_tab
del hsc_wide_tab
del hsc_deep_tab
del mcxc_tab
del planck_tab
del spt_ecs_tab
del spt_sz_tab

# Mark shear footprint

In [90]:
rubin_wide_metadetect_mask = hsp.HealSparseMap.read('/global/cfs/cdirs/lsst/utilities/data-registry/lsst_desc_working/user/smau/rubin_dp2_wide_metadetect_mask_0.2.0.hsp')
rubin_wide_anacal_mask = hsp.HealSparseMap.read('/global/cfs/cdirs/lsst/utilities/data-registry/lsst_desc_working/user/smau/rubin_dp2_wide_anacal_mask_0.2.0.hsp')

In [91]:
in_metadetect_wide = rubin_wide_metadetect_mask.get_values_pos(full_clusters_catalog['RA'], full_clusters_catalog['DEC'])
in_anacal_wide = rubin_wide_anacal_mask.get_values_pos(full_clusters_catalog['RA'], full_clusters_catalog['DEC'])

In [92]:
full_clusters_catalog['in_metadetect_wide'] = in_metadetect_wide
full_clusters_catalog['in_anacal_wide'] = in_anacal_wide

# Get tracts and patches

In [93]:
butler = Butler("dp2", collections="dp2")
skymap = butler.get('skyMap', skymap='lsst_cells_v2')

In [94]:
def get_tract_patch(ra, dec):
    point = geom.SpherePoint(ra * geom.degrees, dec * geom.degrees)
    
    tract_info = skymap.findTract(point)
    tract_id = tract_info.getId()
    patch_info = tract_info.findPatch(point)
    patch_id= patch_info.getSequentialIndex()
    
    return tract_id, patch_id

In [95]:
tracts = []
patches = []
for idx in tqdm.tqdm(range(len(full_clusters_catalog))):
    ra, dec = full_clusters_catalog['RA'][idx], full_clusters_catalog['DEC'][idx]
    tract, patch = get_tract_patch(ra, dec)
    tracts.append(tract)
    patches.append(patch)

100%|██████████| 835277/835277 [00:42<00:00, 19805.82it/s]


In [96]:
full_clusters_catalog['dp2_tract'] = tracts
full_clusters_catalog['dp2_patch'] = patches

In [97]:
# Uncomment this cell to save the full clusters catalog with DP2 tract information to a CSV file:

full_clusters_catalog.write('dp2_cluster_overlap_oct29.csv', format='csv', overwrite=True)

# Save to dataregistry

In [1]:
import os
from dataregistry import DataRegistry

datareg = DataRegistry()

name = "dp2_known_clusters"
version = "0.2.0"
description = """
Known galaxy clusters from 11 external catalogues (ACT DR6, DECADE, DES Y3 redMaPPer, DES Y3 WaZP, HSC Wide/Deep, MCXC-II, Planck, SPT-ECS, SPT SZ 2500d, eRASS1), 835,277 rows, one row per catalogue entry. in_metadetect_wide/in_anacal_wide selects clusters in the DP2 shear footprint. Contains cross-catalogue duplicates. Source: github.com/LSSTDESC/DP2_known_clusters (oct29).
"""

id_0, exec_0 = datareg.register_dataset(
    name,
    version,
    description=description,
    owner_type="user",
    old_location='/global/cfs/cdirs/desc-cl/DP2_known_clusters/dp2_cluster_overlap_oct29.csv',
    is_overwritable=True,
    )